# 04 - Sessions, state and context

**Runtime:** ADK 2.10.0, your existing `dataengenv` kernel and Application Default Credentials (ADC).

**Cost:** Two conversation turns, typically about three Gemini calls including the tool follow-up. Synthetic data only. Run top to bottom; no `course.py`, configuration file or data file is needed.

## Initialize

As in notebooks 01-03, configure the client here. Standard TLS certificate verification is enabled.

In [ ]:
from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

PROJECT = "gen-lang-client-0367558705"
LOCATION = "global"
MODEL = "gemini-2.5-flash-lite"
APP_NAME = "ecommerce_lab"
USER_ID = "customer_101"

model = Gemini(
    model=MODEL,
    client_kwargs={"vertexai": True, "project": PROJECT, "location": LOCATION},
)

## 1. What is a session, and why use one?

A session is one conversation record: identity, event history and state. The model does not independently remember previous calls; the Runner uses the session when preparing later turns.

| Concept | Example |
|---|---|
| Session | One customer's current shopping conversation |
| History (`events`) | "Remember EUR", followed by the agent's answer |
| State | `user:currency = "EUR"`, an explicit application value |
| Session service | Creates, retrieves, lists and stores conversations |
| Tool context | Gives an executing tool access to tracked state |

Reuse the session ID to continue a chat; create a new session for a separate chat. Reads use `app_name`, `user_id` and `session_id` together.

## 2. Compare session storage options

| Service | Survives restart? | When to use |
|---|---|---|
| `InMemorySessionService` | No | Notebooks and tests; one process |
| `DatabaseSessionService` | Yes, with durable storage | SQL-backed applications; SQLite locally or a shared database for workers |
| `VertexAiSessionService` | Yes | Managed sessions on Vertex AI Agent Engine |
| `SqliteSessionService` | Yes, with a saved file | Dedicated local SQLite backend; availability depends on ADK version |

Database services require drivers/configuration; Vertex requires cloud setup and an Agent Engine identifier. This lesson uses **InMemorySessionService** for zero storage setup. See the [official session guide](https://github.com/google/adk-python/blob/main/docs/guides/sessions/session/index.md) for backend-specific configuration.

## 3. Create a session before calling the model

One service object can hold many conversations. Recreating an in-memory service starts a separate, empty store. Keep this instance for the examples below.

In [ ]:
session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    state={
        "customer_id": "C101",
        "user:currency": "INR",
        "app:shop_name": "DemoMart",
    },
)
print("Session ID:", session.id)
print("Initial state:", session.state)
print("Initial events:", len(session.events))

## 4. Choose the state scope

| Key | Scope | Example use |
|---|---|---|
| `customer_id` (no prefix) | This session | Current task details |
| `user:currency` | This user within the app | Preference across chats |
| `app:shop_name` | All users in the app | Shared application setting |
| `temp:preference_changed` | Current invocation only | Intermediate tool flag |

An invocation is one request processed through its response. Shared scopes use the same underlying store; they do not copy conversation history.

Write changes through the injected tool context so ADK records event deltas. Do not expect edits to a fetched `session.state` dictionary to persist. Reassign nested values instead of mutating them in place; store serializable values.

[State scopes and updates](https://github.com/google/adk-docs/blob/main/docs/sessions/state.md).

In [ ]:
from google.adk.tools import ToolContext


def remember_preference(currency: str, tool_context: ToolContext) -> dict:
    """Remember a preferred display currency without converting any prices."""
    if currency not in {"INR", "USD", "EUR"}:
        return {"error": "Unsupported currency"}
    tool_context.state["user:currency"] = currency
    tool_context.state["temp:preference_changed"] = True
    return {"currency": currency}


def get_order(order_id: str) -> dict:
    """Look up a synthetic order's invoice total and original currency."""
    orders = {
        "O1001": {"order_id": "O1001", "total": 4126.46, "currency": "INR"},
        "O1002": {"order_id": "O1002", "total": 1060.82, "currency": "INR"},
    }
    return orders.get(order_id, {"error": "Order not found"})

## 5. Connect the agent and Runner

ADK supplies `tool_context`; the user does not pass it. Instruction placeholders read state values. `output_key` saves the final answer under a session-scoped key, as in notebook 03.

In [ ]:
agent = Agent(
    name="stateful_shop",
    model=model,
    instruction=(
        "Customer {customer_id}; preferred currency {user:currency}. "
        "Use remember_preference when asked to remember currency. "
        "Use get_order for order facts. Never invent exchange rates or convert prices."
    ),
    tools=[remember_preference, get_order],
    output_key="last_answer",
    generate_content_config=types.GenerateContentConfig(
        temperature=0, max_output_tokens=768,
    ),
)
runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)

## 6. Two turns in the same conversation

The first turn saves a preference; the second asks about it. Both use the same session ID. Print state changes as well as answers to observe what the tool actually stores.

In [ ]:
for prompt in [
    "Remember EUR as my preferred display currency.",
    "What currency did I ask you to remember?",
]:
    print("\nUser:", prompt)
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session.id, new_message=message,
    ):
        if event.error_code:
            raise RuntimeError(f"{event.error_code}: {event.error_message}")
        for call in event.get_function_calls():
            print(f"Tool: {call.name}({call.args})")
        if event.actions and event.actions.state_delta:
            print("State changes:", event.actions.state_delta)
        if event.is_final_response() and event.content:
            answer = "".join(part.text or "" for part in event.content.parts or [])
            print("Answer:", answer)

session = await session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=session.id,
)
print("\nSaved state:", session.state)
print("Saved events:", len(session.events))
print("Temporary flag persisted:", "temp:preference_changed" in session.state)

## 7. New chat, same user versus another user

No model calls here. The same user should inherit the saved preference; a different user should not. Both share the shop name. Neither new chat inherits the first chat's history or `customer_id`.

The current agent requires `{customer_id}`. Supply that session-scoped value before using it with a new chat; these examples only inspect storage.

In [ ]:
second_session = await session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID,
)
other_user_session = await session_service.create_session(
    app_name=APP_NAME, user_id="customer_202",
)

for label, current in [
    ("Original chat", session),
    ("New chat, same user", second_session),
    ("New chat, different user", other_user_session),
]:
    print(label)
    print("  Currency:", current.state.get("user:currency", "not set"))
    print("  Customer:", current.state.get("customer_id", "not set"))
    print("  Shop:", current.state.get("app:shop_name"))
    print("  Events:", len(current.events))

assert second_session.state["user:currency"] == session.state["user:currency"]
assert "customer_id" not in second_session.state
assert "user:currency" not in other_user_session.state
assert other_user_session.state["app:shop_name"] == "DemoMart"
assert not second_session.events

## 8. List sessions and demonstrate an empty store

These operations do not call Gemini. The new store below cannot retrieve the original session.

In [ ]:
listed = await session_service.list_sessions(app_name=APP_NAME, user_id=USER_ID)
print("This user's session IDs:", [item.id for item in listed.sessions])

fresh_service = InMemorySessionService()
missing = await fresh_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=session.id,
)
print("Original session in a fresh service:", missing)
assert missing is None

## Try it

Change EUR to USD and rerun the conversation cell. Then rerun the comparison: the same user's new chat should see USD. Try asking for O1001; its invoice remains INR because saving a preference does not convert money.

**Connection to earlier notebooks:** 01 introduced the Runner and session; 02 added tools; 03 saved structured output. Here, tools update scoped state and later turns reuse it.

Configuration and tools are all above. No environment activation, login cells or shared helper files are needed.